In [ ]:
%%configure -f
{"defaultLakehouse": {"name": "lh_fraud"}}

In [ ]:
# Lakehouse layout (app/layout.py): logical name -> schema.table — bronze · quality · silver · gold · ml · mart · serving
import json as _json
T = _json.loads('{"transactions_train": "bronze.transactions_train", "transactions_test": "bronze.transactions_test", "customers": "bronze.customers", "cards": "bronze.cards", "merchants": "bronze.merchants", "digital_events": "bronze.digital_events", "fraud_labels_train": "bronze.fraud_labels_train", "legacy_rule_alerts_train": "bronze.legacy_rule_alerts_train", "v_transactions_all": "bronze.v_transactions_all", "v_transactions_train_labeled": "bronze.v_transactions_train_labeled", "bronze_ingestion_log": "bronze.ingestion_log", "dq_rules": "quality.rules", "dq_issues": "quality.issues", "dq_row_status": "quality.row_status", "dq_summary": "quality.summary", "quarantine_transactions": "quality.quarantine_transactions", "quarantine_customers": "quality.quarantine_customers", "quarantine_cards": "quality.quarantine_cards", "quarantine_merchants": "quality.quarantine_merchants", "quarantine_digital_events": "quality.quarantine_digital_events", "silver_transactions": "silver.transactions", "gold_features": "gold.features", "gold_scores": "gold.scores", "gold_alert_queue": "gold.alert_queue", "gold_alert_explanations": "gold.alert_explanations", "gold_model_metrics": "ml.model_metrics", "gold_model_leaderboard": "ml.model_leaderboard", "gold_model_importance": "ml.model_importance", "gold_model_recall_by_type": "ml.model_recall_by_type", "gold_model_ci": "ml.model_ci", "gold_model_robustness": "ml.model_robustness", "gold_model_robustness_types": "ml.model_robustness_types", "dim_date": "mart.dim_date", "dim_heure": "mart.dim_heure", "dim_geo": "mart.dim_geo", "dim_client": "mart.dim_client", "dim_merchant": "mart.dim_merchant", "dim_channel": "mart.dim_channel", "dim_card": "mart.dim_card", "fact_transactions": "mart.fact_transactions", "fact_digital_events": "mart.fact_digital_events", "pbi_alerts": "serving.pbi_alerts", "pbi_compare": "serving.pbi_compare", "pbi_factors": "serving.pbi_factors", "pbi_importance": "serving.pbi_importance", "pbi_kpi": "serving.pbi_kpi", "pbi_leaderboard": "serving.pbi_leaderboard", "pbi_recall_type": "serving.pbi_recall_type", "pbi_recall_wide": "serving.pbi_recall_wide"}')


# 04 · Robustness: rolling backtest, confidence intervals, model registry
Read-only on the pipeline: it does **not** change `gold_scores` or the alert queue (demo IDs stay frozen).

1. **Rolling-origin backtest**: same features, sampling (all frauds + 10 % legit, seed 42) and LightGBM settings as notebook 02,
   re-trained 4 times, each time only on the past, and tested on the next 2 months (Dec–Jan, Feb–Mar, Apr–May, Jun–Jul).
   Fold 4 is exactly the production set-up (train ≤ May 2026, test Jun–Jul).
2. **Day-block bootstrap** (1,000 resamples of whole days) on the production validation scores → 95 % confidence intervals.
3. **Agreement check** between the fold-4 model and the scores in `gold_scores`, then **MLflow** experiment + registered model
   `hyson-fraud-lightgbm`.

Outputs: `gold_model_robustness`, `gold_model_robustness_types`, `gold_model_ci`, `Files/robustness/summary.json`,
`Files/robustness/runtime_versions.json`, MLflow experiment `hyson-fraud-robustness`, ML model `hyson-fraud-lightgbm`.

In [ ]:
import json, platform, sys, time
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from sklearn.metrics import average_precision_score
import lightgbm as lgb
import mlflow
mlflow.autolog(disable=True)          # only the runs created below

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
LEGIT_FRAC, SEED, N_BOOT = 0.10, 42, 1000
FOLDS = [("F1 · déc.–janv.", "2025-12-01", "2026-02-01"), ("F2 · févr.–mars", "2026-02-01", "2026-04-01"),
         ("F3 · avr.–mai", "2026-04-01", "2026-06-01"), ("F4 · juin–juil. (validation)", "2026-06-01", "2026-08-01")]
t0 = time.time()

g = spark.table(T["gold_features"])
tr = g.filter(F.col("period") == "train")
# training pool = exactly notebook 02's sample (all frauds + 10 % of legit train rows, seed 42)
pool = (tr.filter(F.col("is_fraud") == 1).withColumn("sample_w", F.lit(1.0))
          .unionByName(tr.filter(F.col("is_fraud") == 0).sample(fraction=LEGIT_FRAC, seed=SEED)
                         .withColumn("sample_w", F.lit(1.0 / LEGIT_FRAC)))).toPandas()
# test windows = every labelled row (no sampling) from the first fold on
windows = g.filter(F.col("period").isin("train", "valid") & (F.col("timestamp") >= F.lit(FOLDS[0][1]))).toPandas()

def naive(s):
    s = pd.to_datetime(s)
    return s.dt.tz_localize(None) if getattr(s.dt, "tz", None) is not None else s

for d in (pool, windows):
    d["timestamp"] = naive(d["timestamp"])
pool = pool.sort_values(["period", "timestamp"]).reset_index(drop=True)      # same order as notebook 02
windows = windows.sort_values("timestamp").reset_index(drop=True)
print("pool:", len(pool), "rows,", int(pool.is_fraud.sum()), "frauds | windows:", len(windows), "rows |", round(time.time() - t0), "s")

In [ ]:
CATS = ["channel", "entry_mode", "transaction_type", "merchant_category", "segment", "kyc_risk_level",
        "card_type", "merchant_type", "decline_reason"]
LEAKY = {"f_merchant_risk", "f_device_risk", "f_beneficiary_risk"}   # target-encoded on all train labels: never used
fcols = [c for c in pool.columns if c.startswith("f_")]
FEATS = [c for c in fcols if c not in LEAKY] + CATS
assert not any(k in c for c in FEATS for k in ["is_fraud", "scenario", "reported", "gender"]), "leak column in features"

# category dictionaries identical to notebook 02 (train sample + valid + test values)
other = g.filter(F.col("period") != "train")
CAT_VALUES = {}
for c in CATS:
    seen = {r[0] for r in other.select(F.col(c).cast("string")).distinct().collect() if r[0] is not None}
    CAT_VALUES[c] = sorted(seen | set(pool[c].dropna().astype(str)))

def prep(d):
    d[fcols] = d[fcols].astype("float32").replace([np.inf, -np.inf], np.nan)
    d["is_fraud"] = d["is_fraud"].fillna(0).astype(int)
    X = d[FEATS].copy()
    for c in CATS:
        X[c] = pd.Categorical(X[c].astype("string"), categories=CAT_VALUES[c])
    return X

Xp, Xw = prep(pool), prep(windows)
legacy_ids = set(spark.table(T["legacy_rule_alerts_train"]).select("transaction_id").distinct().toPandas().transaction_id)
windows["legacy_alert"] = windows.transaction_id.isin(legacy_ids).astype(int)
LGB_P = dict(n_estimators=600, learning_rate=0.03, num_leaves=63, min_child_samples=50, subsample=0.8,
             subsample_freq=1, colsample_bytree=0.7, reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbose=-1)
print(len(FEATS), "features")

In [ ]:
def top_k(y, s, k):
    idx = np.argsort(-s, kind="stable")[:k]
    tp = int(y[idx].sum())
    return tp / max(k, 1), tp / max(int(y.sum()), 1), idx

rows, type_rows, models, f4 = [], [], {}, None
mlflow.set_experiment("hyson-fraud-robustness")
parent = mlflow.start_run(run_name="rolling-backtest")
for name, start, end in FOLDS:
    t1 = time.time()
    trm = (pool.timestamp < pd.Timestamp(start)).values
    tem = ((windows.timestamp >= pd.Timestamp(start)) & (windows.timestamp < pd.Timestamp(end))).values
    y_tr, sw = pool.loc[trm, "is_fraud"].values, pool.loc[trm, "sample_w"].values
    ratio = sw[y_tr == 0].sum() / max(y_tr.sum(), 1)                      # real imbalance (sampling undone)
    m = lgb.LGBMClassifier(**LGB_P).fit(Xp[trm], y_tr, sample_weight=sw * np.where(y_tr == 1, min(ratio, 50.0), 1.0))
    v = windows[tem]
    y, s = v.is_fraud.values, m.predict_proba(Xw[tem])[:, 1]
    k = max(int(len(v) * 0.01), 1)
    p1, r1, idx = top_k(y, s, k)
    k_leg = int(v.legacy_alert.sum())
    leg_tp = int(((v.legacy_alert == 1) & (v.is_fraud == 1)).sum())
    _, r_same, _ = top_k(y, s, k_leg) if k_leg else (np.nan, np.nan, None)
    amt = np.nan_to_num(v.f_amount.values.astype(float))
    imp = pd.Series(m.booster_.feature_importance("gain"), index=FEATS).nlargest(8)
    row = {"fold": name, "train_to": start, "test_from": start, "test_to": str((pd.Timestamp(end) - pd.Timedelta(days=1)).date()),
           "train_rows": int(trm.sum()), "train_frauds": int(y_tr.sum()), "test_rows": int(len(v)), "test_frauds": int(y.sum()),
           "pr_auc": round(float(average_precision_score(y, s)), 4), "top1_alerts": k,
           "top1_precision": round(p1, 4), "top1_recall": round(r1, 4),
           "value_recall": round(float(amt[idx][y[idx] == 1].sum() / max(amt[y == 1].sum(), 1e-9)), 4),
           "legacy_alerts": k_leg, "legacy_precision": round(leg_tp / max(k_leg, 1), 4),
           "legacy_recall": round(leg_tp / max(int(y.sum()), 1), 4), "same_volume_recall": round(float(r_same), 4),
           "top_features": " | ".join(imp.index)}
    rows.append(row)
    caught = set(v.transaction_id.values[idx])
    fr = v[v.is_fraud == 1]
    for sc, grp in fr.groupby("fraud_scenario"):
        type_rows.append({"fold": name, "fraud_scenario": sc, "frauds": int(len(grp)),
                          "caught_model": int(grp.transaction_id.isin(caught).sum()), "caught_legacy": int(grp.legacy_alert.sum())})
    with mlflow.start_run(run_name=name, nested=True):
        mlflow.log_params({"train_to": start, "test_from": start, "test_to": row["test_to"], **{f"lgb_{k2}": v2 for k2, v2 in LGB_P.items()}})
        mlflow.log_metrics({k2: float(row[k2]) for k2 in ["pr_auc", "top1_precision", "top1_recall", "value_recall",
                                                         "legacy_precision", "legacy_recall", "same_volume_recall"]})
    models[name] = m
    if name.startswith("F4"):
        f4 = pd.DataFrame({"transaction_id": v.transaction_id.values, "s": s, "is_fraud": y, "legacy_alert": v.legacy_alert.values,
                           "timestamp": v.timestamp.values})
    print(f"{name}: PR-AUC {row['pr_auc']} · top1% P {row['top1_precision']:.3f} R {row['top1_recall']:.3f} · "
          f"legacy P {row['legacy_precision']:.3f} R {row['legacy_recall']:.3f} · {round(time.time() - t1)} s")

rob = pd.DataFrame(rows)
summary = {c: {"mean": round(float(rob[c].mean()), 4), "std": round(float(rob[c].std(ddof=1)), 4),
               "min": round(float(rob[c].min()), 4), "max": round(float(rob[c].max()), 4)}
           for c in ["pr_auc", "top1_precision", "top1_recall", "value_recall", "legacy_precision", "legacy_recall"]}
mlflow.log_metrics({f"{c}_mean": v2["mean"] for c, v2 in summary.items()} | {f"{c}_std": v2["std"] for c, v2 in summary.items()})
print(rob.drop(columns="top_features").to_string(index=False))

In [ ]:
# agreement between the fold-4 model and the production scores used by the app
prod = spark.table(T["gold_scores"]).filter(F.col("period") == "valid").select("transaction_id", "score_model").toPandas()
a = f4.merge(prod, on="transaction_id")
k = max(int(len(a) * 0.01), 1)
agree = {"rows_matched": int(len(a)),
         "spearman": round(float(a.s.rank().corr(a.score_model.rank())), 6),
         "top1_overlap": round(len(set(a.nlargest(k, "s").transaction_id) & set(a.nlargest(k, "score_model").transaction_id)) / k, 4),
         "max_abs_diff": float((a.s - a.score_model).abs().max())}
print("agreement with gold_scores:", agree)

# day-block bootstrap on the production validation scores (resample whole days: errors are correlated within a day)
b = a.copy()
b["day"] = pd.to_datetime(b.timestamp).dt.floor("D")
blocks = list(b.groupby("day").indices.values())
Y, S, LG = b.is_fraud.values, b.score_model.values, b.legacy_alert.values
def stats(ix):
    y, s, lg = Y[ix], S[ix], LG[ix]
    k = max(int(len(ix) * 0.01), 1)
    top = np.argpartition(-s, k - 1)[:k]
    tp, n_fr, k_leg = int(y[top].sum()), max(int(y.sum()), 1), max(int(lg.sum()), 1)
    leg_tp = int((lg & y).sum())
    return {"top1_precision": tp / k, "top1_recall": tp / n_fr, "pr_auc": average_precision_score(y, s),
            "legacy_precision": leg_tp / k_leg, "legacy_recall": leg_tp / n_fr,
            "recall_gain_points": (tp - leg_tp) / n_fr, "false_alerts_avoided": (int(lg.sum()) - leg_tp) - (k - tp)}
point = stats(np.arange(len(b)))
rng = np.random.default_rng(SEED)
boot = pd.DataFrame([stats(np.concatenate([blocks[i] for i in rng.integers(0, len(blocks), len(blocks))])) for _ in range(N_BOOT)])
ci = pd.DataFrame([{"metric": m2, "estimate": round(float(point[m2]), 4), "ci_low": round(float(boot[m2].quantile(0.025)), 4),
                    "ci_high": round(float(boot[m2].quantile(0.975)), 4), "method": f"day-block bootstrap, {N_BOOT} resamples, 95 %",
                    "period": "validation Jun–Jul 2026"} for m2 in point])
print(ci.to_string(index=False))

In [ ]:
def save(df, name):
    spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(T.get(name, name))

save(rob, "gold_model_robustness")
save(pd.DataFrame(type_rows), "gold_model_robustness_types")
save(ci, "gold_model_ci")

versions = {"python": platform.python_version(), "lightgbm": lgb.__version__, "mlflow": mlflow.__version__,
            "pandas": pd.__version__, "numpy": np.__version__}
for mod in ["sklearn", "shap", "scipy", "pyarrow", "sempy", "sempy_labs"]:
    try:
        versions[mod] = __import__(mod).__version__
    except Exception:
        pass
notebookutils.fs.put("Files/ml/robustness/runtime_versions.json", json.dumps(versions, indent=1), True)

# MLflow: production-configuration model (fold 4) with its card, registered as an ML model item
registered = None
try:
    with mlflow.start_run(run_name="production-lightgbm", nested=True):
        mlflow.log_params({"features": len(FEATS), "train_to": "2026-06-01", "legit_sample": LEGIT_FRAC, "seed": SEED,
                           "alert_budget": "top 1 % per period", **{f"lgb_{k2}": v2 for k2, v2 in LGB_P.items()}})
        mlflow.log_metrics({f"valid_{r.metric}": float(r.estimate) for r in ci.itertuples()} |
                           {f"agreement_{k2}": float(v2) for k2, v2 in agree.items()})
        mlflow.log_dict({"features": FEATS, "categoricals": CATS, "excluded_leaky": sorted(LEAKY)}, "features.json")
        mlflow.log_text(
            "# Model card — hyson-fraud-lightgbm\n"
            "Purpose: rank card / digital-banking operations by fraud risk for a fraud analyst (top 1 % = alert queue).\n"
            "Data: synthetic Tunisian retail bank, gold_features (73 past-only features), trained Jan 2025 → May 2026.\n"
            f"Validation Jun–Jul 2026: PR-AUC {point['pr_auc']:.3f}, top-1 % precision {point['top1_precision']:.1%}, "
            f"recall {point['top1_recall']:.1%} (legacy rules: {point['legacy_precision']:.1%} / {point['legacy_recall']:.1%}).\n"
            f"Rolling backtest (4 folds): top-1 % recall {summary['top1_recall']['mean']:.1%} ± {summary['top1_recall']['std']:.1%}.\n"
            "Not used as features: gender, IDs, fraud_scenario, fraud_reported_date (except frauds already reported), "
            "target-encoded entity risk. Decision stays with the analyst; the LLM only explains.\n"
            "Limits: synthetic data, batch scoring, label delay not simulated in the backtest.\n", "model_card.md")
        info = mlflow.lightgbm.log_model(models[FOLDS[-1][0]], artifact_path="model", registered_model_name="hyson-fraud-lightgbm")
        registered = {"model_uri": info.model_uri, "name": "hyson-fraud-lightgbm"}
except Exception as exc:
    registered = {"error": f"{type(exc).__name__}: {str(exc)[:300]}"}
mlflow.end_run()

out = {"folds": rows, "summary": summary, "agreement": agree, "ci": ci.to_dict("records"), "versions": versions,
       "registered": registered, "seconds": round(time.time() - t0)}
notebookutils.fs.put("Files/ml/robustness/summary.json", json.dumps(out, indent=1, default=str), True)
print("=== CHECK 04 ===", json.dumps({"summary": summary, "agreement": agree, "registered": registered}, default=str))

In [ ]:
# free the capacity when run by hand (a scheduled/API job ends its own session)
if str(notebookutils.runtime.context.get("isForInteractive")).lower() == "true":
    mssparkutils.session.stop()